# Mini GR00T + SONIC: BONES curriculum on Colab

This notebook mounts Google Drive, clones `YigitGunduc/robot`, extracts a small SONIC-style filename-filtered subset from BONES-SEED, preprocesses it, runs a CUDA/MuJoCo-Warp smoke test, and trains a two-stage body-controller curriculum. The optional final section collects replay and trains the text-to-token flow model.

Before starting, select **Runtime → Change runtime type → NVIDIA GPU**. Push the notebook and associated code changes to the repository before cloning from a fresh Colab runtime.

In [ ]:
from pathlib import Path

# Paths supplied by the Drive layout in the prompt.
DRIVE_BONES = Path('/content/drive/MyDrive/Datasets/bones-seed')
DRIVE_WORK = Path('/content/drive/MyDrive/mini_groot_sonic_colab')
REPO_URL = 'https://github.com/YigitGunduc/robot.git'  # HTTPS works in Colab without your Mac SSH key.
REPO_BRANCH = 'master'
REPO_DIR = Path('/content/robot')
MENAGERIE_DIR = Path('/content/mujoco_menagerie')

SEED = 0
STAGE1_RECORDS = 24       # stand / idle / basic walking
STAGE2_RECORDS = 64       # stage 1 plus walk / jog / run / simple turns
NUM_ENVS = 64             # raise to 128 only after the smoke test is stable
STAGE1_ITERATIONS = 1000
STAGE2_TOTAL_ITERATIONS = 3000  # absolute iteration, so stage 2 adds about 2000
VISUAL_ROLLOUTS = 3       # held-out side-by-side policy/reference MP4 files
VIDEO_MAX_STEPS = 500     # 10 seconds at the 50 Hz controller rate
REPLAY_EPISODES = 48
FLOW_EPOCHS = 15

COPY_ARCHIVE_TO_LOCAL = False  # True is faster if the archive and extracted files fit /content.
FORCE_RAW_SELECTION = False
FORCE_PREPROCESS = False
RESUME_EXISTING_RUNS = True
RUN_BODY_TRAINING = True
RUN_REPLAY_AND_FLOW = False    # Enable only after held-out body tracking is acceptable.

EASY_KEYWORDS = ('stand', 'standing', 'idle', 'walk', 'walking')
ADVANCED_KEYWORDS = ('jog', 'jogging', 'run', 'running', 'turn', 'turning')
EXPANDED_KEYWORDS = EASY_KEYWORDS + ADVANCED_KEYWORDS
# SONIC's upstream filename denylist is imported from the repository after installation.


In [ ]:
import shutil
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')
assert DRIVE_BONES.is_dir(), f'BONES directory not found: {DRIVE_BONES}'
assert (DRIVE_BONES / 'metadata').is_dir(), 'BONES metadata directory is missing'
archives = [DRIVE_BONES / 'g1.tar.zst', DRIVE_BONES / 'g1.tar.gz']
assert any(path.exists() for path in archives), 'Neither g1.tar.zst nor g1.tar.gz was found'
DRIVE_WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)


## Clone the code and install dependencies

For a private GitHub repository, replace `REPO_URL` with an authenticated HTTPS URL or configure a Colab secret. Do not paste a long-lived token into a saved notebook.

In [ ]:
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git checkout; remove or rename it')
else:
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)
    ], check=True)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[all]', 'pyarrow'
], check=True)

# Editable installs are exposed through a .pth file that this already-running
# Colab kernel will not re-read until restart. Add src now so imports work in
# the very next cell while new subprocesses continue using the installation.
import importlib

source_root = str(REPO_DIR / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)
importlib.invalidate_caches()
import mini_groot_sonic

print('Imported mini_groot_sonic from', mini_groot_sonic.__file__)
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=True)


## Fetch and validate the 29-DOF G1 MJCF

The project intentionally does not duplicate robot meshes. This cell obtains the maintained Unitree G1 model from MuJoCo Menagerie and validates the exact joint/actuator contract before preprocessing.

In [ ]:
MENAGERIE_URL = 'https://github.com/google-deepmind/mujoco_menagerie.git'
if not (MENAGERIE_DIR / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', MENAGERIE_URL, str(MENAGERIE_DIR)], check=True)
MJCF = MENAGERIE_DIR / 'unitree_g1' / 'scene.xml'
assert MJCF.exists(), f'G1 scene not found: {MJCF}'

# Menagerie attaches the head mesh directly to torso_link, so use the torso body
# as the stable head proxy for keypoint rewards/goals. Keep this override local
# to Colab rather than changing assumptions for other G1 MJCF variants.
import yaml

COLAB_CONFIG = Path('/content/mini_groot_sonic_colab.yaml')
config_values = yaml.safe_load((REPO_DIR / 'configs/default.yaml').read_text())
config_values.setdefault('sim', {})['keypoint_body_names'] = [
    'torso_link', 'left_wrist_yaw_link', 'right_wrist_yaw_link',
    'left_ankle_roll_link', 'right_ankle_roll_link',
]
COLAB_CONFIG.write_text(yaml.safe_dump(config_values, sort_keys=False))

import mujoco

from mini_groot_sonic.config import load_project_config
from mini_groot_sonic.sim.g1_mapping import G1ModelMap

cfg = load_project_config(COLAB_CONFIG)
model = mujoco.MjModel.from_xml_path(str(MJCF))
mapping = G1ModelMap.from_mjmodel(
    model, cfg.sim.root_body_name, tuple(cfg.sim.keypoint_body_names)
)
kind = 'position' if mapping.actuator_is_position.all() else 'motor/PD torque'
print(f'MJCF={MJCF}')
print(f'nq={model.nq}, nv={model.nv}, actuated_joints={len(mapping.joint_names)}, actuator_mode={kind}')
print(mapping.joint_names)


## Select and extract only basic BONES motions

This applies SONIC's offline denylist and the small curriculum allowlists to archive paths/filenames only; captions and metadata never decide admission. Stage 1 contains 24 stand/idle/walk files, and stage 2 adds enough jog/run/turn files to reach 64. Whole source motions are kept (no temporal-label expansion), and the exact selection is cached on Drive. Object-interaction filenames are intentionally not excluded yet.

In [ ]:
import json
import os
import random

from mini_groot_sonic.data.bones import (
    SONIC_DEFAULT_FILTER_KEYWORDS,
    sonic_filename_allowed,
)

SONIC_FILTER_KEYWORDS = SONIC_DEFAULT_FILTER_KEYWORDS
selection_name = f'sonic_filename_v3_seed{SEED}_s1{STAGE1_RECORDS}_s2{STAGE2_RECORDS}'
RAW_CACHE = DRIVE_WORK / 'cache' / selection_name
RAW_SELECTED = Path('/content/bones_selected_csv')
MANIFEST_PATH = RAW_CACHE / 'selection.json'

if FORCE_RAW_SELECTION and RAW_CACHE.exists():
    shutil.rmtree(RAW_CACHE)
if RAW_SELECTED.exists():
    shutil.rmtree(RAW_SELECTED)
RAW_SELECTED.mkdir(parents=True)

if MANIFEST_PATH.exists() and list(RAW_CACHE.glob('*.csv')):
    print('Using cached SONIC-style filename selection from Drive')
    shutil.copytree(RAW_CACHE, RAW_SELECTED, dirs_exist_ok=True, ignore=shutil.ignore_patterns('selection.json'))
    manifest = json.loads(MANIFEST_PATH.read_text())
else:
    archive = next(path for path in archives if path.exists())
    if COPY_ARCHIVE_TO_LOCAL:
        local_archive = Path('/content') / archive.name
        if not local_archive.exists() or local_archive.stat().st_size != archive.stat().st_size:
            print(f'Copying {archive.name} to local scratch...')
            shutil.copy2(archive, local_archive)
        archive = local_archive

    list_cmd = ['tar']
    if archive.suffix == '.zst':
        list_cmd += ['--zstd']
    list_cmd += ['-tf', str(archive)]
    print(f'Scanning archive members from {archive} ...')
    listing = subprocess.run(list_cmd, check=True, capture_output=True, text=True).stdout.splitlines()
    csv_members = [name for name in listing if name.lower().endswith('.csv')]
    member_by_stem = {}
    for member in csv_members:
        member_by_stem.setdefault(Path(member).stem, member)
    print(f'Archive contains {len(csv_members):,} CSV members and {len(member_by_stem):,} unique stems')

    def candidates_for(keywords):
        return [
            stem for stem, member in member_by_stem.items()
            if sonic_filename_allowed(member, keywords, SONIC_FILTER_KEYWORDS)
        ]

    rng = random.Random(SEED)
    easy = sorted(set(candidates_for(EASY_KEYWORDS)))
    rng.shuffle(easy)
    stage1_stems = easy[:STAGE1_RECORDS]
    advanced = sorted(set(candidates_for(ADVANCED_KEYWORDS)) - set(stage1_stems))
    random.Random(SEED + 1).shuffle(advanced)
    needed = max(0, STAGE2_RECORDS - len(stage1_stems))
    expansion = advanced[:needed]
    if len(expansion) < needed:
        easy_fallback = sorted(set(easy) - set(stage1_stems) - set(expansion))
        random.Random(SEED + 2).shuffle(easy_fallback)
        expansion += easy_fallback[:needed - len(expansion)]
    stage2_stems = stage1_stems + expansion
    assert stage1_stems, 'No stand/walk candidates matched the BONES filenames'
    assert len(stage2_stems) > len(stage1_stems), 'No run/jog/turn filename candidates matched'

    selected_members = [member_by_stem[stem] for stem in stage2_stems]
    member_file = Path('/content/selected_bones_members.txt')
    member_file.write_text('\n'.join(selected_members) + '\n')
    extract_root = Path('/content/bones_extract')
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir()
    extract_cmd = ['tar']
    if archive.suffix == '.zst':
        extract_cmd += ['--zstd']
    extract_cmd += ['-xf', str(archive), '-C', str(extract_root), '-T', str(member_file)]
    print(f'Extracting only {len(selected_members)} selected CSV files...')
    subprocess.run(extract_cmd, check=True)
    for stem, member in zip(stage2_stems, selected_members):
        relative = member.removeprefix('./')
        source = extract_root / relative
        assert source.exists(), f'Archive member was not extracted: {member}'
        shutil.copy2(source, RAW_SELECTED / f'{stem}.csv')

    manifest = {
        'seed': SEED, 'selection_method': 'sonic_filename_filter',
        'temporal_segments': False,
        'easy_keywords': EASY_KEYWORDS, 'advanced_keywords': ADVANCED_KEYWORDS,
        'expanded_keywords': EXPANDED_KEYWORDS,
        'exclude_keywords': SONIC_FILTER_KEYWORDS,
        'stage1_stems': stage1_stems, 'stage2_stems': stage2_stems,
    }
    RAW_CACHE.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RAW_SELECTED, RAW_CACHE, dirs_exist_ok=True)
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))

def make_stage_root(name, selected_stems):
    root = Path('/content') / f'bones_{name}'
    if root.exists():
        shutil.rmtree(root)
    csv_dir = root / 'g1' / 'csv' / 'selected'
    csv_dir.mkdir(parents=True)
    os.symlink(DRIVE_BONES / 'metadata', root / 'metadata', target_is_directory=True)
    for stem in selected_stems:
        source = RAW_SELECTED / f'{stem}.csv'
        target = csv_dir / source.name
        try:
            os.link(source, target)
        except OSError:
            shutil.copy2(source, target)
    return root

BONES_STAGE1_ROOT = make_stage_root('stage1', manifest['stage1_stems'])
BONES_STAGE2_ROOT = make_stage_root('stage2', manifest['stage2_stems'])
print(f"Stage 1 raw records: {len(manifest['stage1_stems'])}")
print(f"Stage 2 raw records: {len(manifest['stage2_stems'])}")
print('First stage-1 IDs:', manifest['stage1_stems'][:10])


## Preprocess the two curriculum stages

Preprocessing runs MuJoCo FK once on CPU and stores compact 50 Hz tracks. Results are cached on Drive and copied to local scratch on later sessions. To mirror SONIC's offline file filtering, each selected source motion remains one whole training clip; temporal metadata does not expand it into unrelated annotated segments.

In [ ]:
PREPROCESS_CACHE = DRIVE_WORK / 'preprocessed' / selection_name
LOCAL_PREPROCESS = Path('/content/mgsp_preprocessed')

def preprocess_stage(name, bones_root, limit, include_keywords):
    local_out = LOCAL_PREPROCESS / name
    drive_out = PREPROCESS_CACHE / name
    if local_out.exists():
        shutil.rmtree(local_out)
    if not FORCE_PREPROCESS and drive_out.exists() and list(drive_out.glob('*.npz')):
        print(f'Copying cached {name} preprocessing from Drive...')
        shutil.copytree(drive_out, local_out)
    else:
        local_out.mkdir(parents=True)
        cmd = [
            sys.executable, '-m', 'mini_groot_sonic.tools.preprocess_bones',
            '--bones-root', str(bones_root), '--mjcf', str(MJCF), '--out', str(local_out),
            '--limit', str(limit), '--seed', str(SEED),
            '--include-keywords', ','.join(include_keywords),
            '--exclude-keywords', ','.join(SONIC_FILTER_KEYWORDS),
            '--no-temporal-segments',
            '--min-clip-seconds', '1.0',
        ]
        subprocess.run(cmd, cwd=REPO_DIR, check=True)
        if drive_out.exists():
            shutil.rmtree(drive_out)
        shutil.copytree(local_out, drive_out)
    clips = sorted(local_out.glob('*.npz'))
    assert clips, f'No clips produced for {name}'
    print(f'{name}: {len(clips)} preprocessed training clips')
    return local_out

STAGE1_DATA = preprocess_stage('stage1', BONES_STAGE1_ROOT, STAGE1_RECORDS, EASY_KEYWORDS)
STAGE2_DATA = preprocess_stage('stage2', BONES_STAGE2_ROOT, STAGE2_RECORDS, EXPANDED_KEYWORDS)


## Tests and target-GPU smoke test

Do not start PPO unless this cell passes. It catches dependency, MJCF, CUDA interop, reset, reference-shape, and actuator-boundary failures on the actual Colab GPU.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=REPO_DIR, check=True)

import torch

from mini_groot_sonic.data.motion_bank import MotionBank
from mini_groot_sonic.sim.mjwarp_env import MJWarpG1VecEnv

assert torch.cuda.is_available(), 'CUDA is unavailable; choose a GPU Colab runtime'
cfg = load_project_config(COLAB_CONFIG)
cfg.sim.mjcf = MJCF
cfg.sim.device = 'cuda:0'
smoke_paths = sorted(STAGE1_DATA.glob('*.npz'))[:2]
for randomized in (False, True):
    cfg.sim.enable_randomization = randomized
    bank = MotionBank(smoke_paths, cfg.sonic, cfg.sim.device)
    env = MJWarpG1VecEnv(cfg.sim, cfg.sonic, len(smoke_paths))
    ids = torch.arange(len(smoke_paths), device=cfg.sim.device)
    frames = torch.zeros(len(smoke_paths), dtype=torch.long, device=cfg.sim.device)
    ref = bank.current_reference(ids, frames)
    obs = env.reset(
        ref['root_pos'], ref['root_quat'], ref['joint_pos'],
        ref['root_linvel'], ref['root_angvel'], ref['joint_vel'],
    )
    obs = env.step(torch.zeros(len(smoke_paths), cfg.sonic.dof, device=cfg.sim.device))
    future = bank.future_reference(ids, frames)
    assert future.shape == (len(smoke_paths), cfg.sonic.future_frames, cfg.sonic.reference_frame_dim)
    assert torch.isfinite(obs.joint_pos).all() and torch.isfinite(obs.root_pos).all()
    print(f'CUDA/MJWarp smoke passed: randomized={randomized}, actuator={env.actuator_mode}, future={tuple(future.shape)}')
    del env, bank, obs, future
torch.cuda.empty_cache()


## Stage 1 — stand, idle, and basic walk

Stage 1 deliberately disables domain randomization. The goal is first to prove that the small controller can track simple motions. Checkpoints go directly to Drive. Re-running the cell resumes the newest checkpoint.

In [ ]:
def latest_numbered_checkpoint(directory):
    paths = list(Path(directory).glob('body_[0-9]*.pt'))
    return max(paths, key=lambda path: int(path.stem.rsplit('_', 1)[1])) if paths else None

BODY_STAGE1 = DRIVE_WORK / 'runs' / selection_name / 'body_stage1'
BODY_STAGE1.mkdir(parents=True, exist_ok=True)
if RUN_BODY_TRAINING:
    resume = latest_numbered_checkpoint(BODY_STAGE1) if RESUME_EXISTING_RUNS else None
    cmd = [
        sys.executable, '-u', '-m', 'mini_groot_sonic.tools.train_body',
        '--motions', str(STAGE1_DATA), '--config', str(COLAB_CONFIG),
        '--mjcf', str(MJCF), '--device', 'cuda:0',
        '--num-envs', str(NUM_ENVS), '--iterations', str(STAGE1_ITERATIONS),
        '--max-motions', str(max(STAGE1_RECORDS, len(list(STAGE1_DATA.glob('*.npz'))))),
        '--out', str(BODY_STAGE1), '--no-randomization',
    ]
    if resume is not None:
        print(f'Resuming Stage 1 from {resume.name}', flush=True)
        cmd += ['--resume', str(resume)]
    else:
        print('Starting Stage 1 from scratch', flush=True)
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
print('Stage-1 checkpoints:', BODY_STAGE1)


## Stage 2 — add jog, run, and simple turns

This resumes stage 1, refreshes reference normalization for the expanded dataset, enables domain randomization, and resets best-checkpoint comparison because stage-2 validation is harder. Later reruns resume within stage 2 without resetting its best score.

In [ ]:
BODY_STAGE2 = DRIVE_WORK / 'runs' / selection_name / 'body_stage2'
BODY_STAGE2.mkdir(parents=True, exist_ok=True)
if RUN_BODY_TRAINING:
    stage2_resume = latest_numbered_checkpoint(BODY_STAGE2) if RESUME_EXISTING_RUNS else None
    transitioning = stage2_resume is None
    if transitioning:
        stage2_resume = BODY_STAGE1 / 'body_best.pt'
        if not stage2_resume.exists():
            stage2_resume = latest_numbered_checkpoint(BODY_STAGE1)
    assert stage2_resume is not None and stage2_resume.exists(), 'Stage-1 checkpoint is missing'
    cmd = [
        sys.executable, '-u', '-m', 'mini_groot_sonic.tools.train_body',
        '--motions', str(STAGE2_DATA), '--config', str(COLAB_CONFIG),
        '--mjcf', str(MJCF), '--device', 'cuda:0',
        '--num-envs', str(NUM_ENVS), '--iterations', str(STAGE2_TOTAL_ITERATIONS),
        '--max-motions', str(max(STAGE2_RECORDS, len(list(STAGE2_DATA.glob('*.npz'))))),
        '--out', str(BODY_STAGE2), '--resume', str(stage2_resume),
    ]
    if transitioning:
        cmd.append('--reset-best')
    print(f'Resuming Stage 2 from {stage2_resume}', flush=True)
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
print('Stage-2 checkpoints:', BODY_STAGE2)


## Reproduce the held-out split and evaluate

The evaluation directory contains only actor/source-disjoint validation clips produced by the same deterministic split as training.

In [ ]:
from mini_groot_sonic.training.utils import split_motion_paths

all_paths = sorted(STAGE2_DATA.glob('*.npz'))
random.Random(SEED).shuffle(all_paths)
all_paths = all_paths[:max(STAGE2_RECORDS, len(all_paths))]
_, validation_paths = split_motion_paths(all_paths, 0.1, SEED)
HELD_OUT = Path('/content/mgsp_held_out')
if HELD_OUT.exists():
    shutil.rmtree(HELD_OUT)
HELD_OUT.mkdir()
for path in validation_paths:
    os.symlink(path, HELD_OUT / path.name)

BODY_CHECKPOINT = BODY_STAGE2 / 'body_best.pt'
if not BODY_CHECKPOINT.exists():
    BODY_CHECKPOINT = latest_numbered_checkpoint(BODY_STAGE2)
assert BODY_CHECKPOINT is not None and BODY_CHECKPOINT.exists(), 'No stage-2 body checkpoint found'
subprocess.run([
    sys.executable, '-m', 'mini_groot_sonic.tools.eval_body',
    '--motions', str(HELD_OUT), '--config', str(COLAB_CONFIG),
    '--mjcf', str(MJCF), '--body', str(BODY_CHECKPOINT),
    '--device', 'cuda:0', '--max-motions', str(max(1, len(validation_paths))),
], cwd=REPO_DIR, check=True)

# Render multiple held-out rollouts side-by-side against their BONES reference.
# MP4 files and JSON metrics sidecars persist on Drive.
import numpy as np
from IPython.display import Video, display


def stored_caption(path):
    with np.load(path, allow_pickle=True) as clip:
        return str(clip['caption'].item()).lower()

# Prefer representative easy walk/stand and faster run/turn clips when the
# held-out partition contains them, then fill remaining slots deterministically.
visual_paths = []
for keywords in (('stand', 'idle'), ('walk',), ('run', 'jog'), ('turn',)):
    match = next((p for p in validation_paths if p not in visual_paths and any(k in stored_caption(p) for k in keywords)), None)
    if match is not None:
        visual_paths.append(match)
for path in validation_paths:
    if len(visual_paths) >= VISUAL_ROLLOUTS:
        break
    if path not in visual_paths:
        visual_paths.append(path)
visual_paths = visual_paths[:VISUAL_ROLLOUTS]
VISUAL_HELD_OUT = Path('/content/mgsp_visual_held_out')
if VISUAL_HELD_OUT.exists():
    shutil.rmtree(VISUAL_HELD_OUT)
VISUAL_HELD_OUT.mkdir()
for index, path in enumerate(visual_paths):
    os.symlink(path, VISUAL_HELD_OUT / f'{index:02d}_{path.name}')

VIDEO_DIR = DRIVE_WORK / 'videos' / selection_name
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
video_paths = []
for motion_index in range(len(visual_paths)):
    video_path = VIDEO_DIR / f'held_out_{motion_index:02d}.mp4'
    subprocess.run([
        sys.executable, '-m', 'mini_groot_sonic.tools.render_body',
        '--motions', str(VISUAL_HELD_OUT), '--config', str(COLAB_CONFIG),
        '--mjcf', str(MJCF), '--body', str(BODY_CHECKPOINT),
        '--device', 'cuda:0', '--motion-index', str(motion_index),
        '--max-steps', str(VIDEO_MAX_STEPS), '--out', str(video_path),
    ], cwd=REPO_DIR, check=True)
    video_paths.append(video_path)
    display(Video(str(video_path), embed=True, width=960))
print('Saved visual evaluations to:', VIDEO_DIR)


## Optional: replay collection and compact GR00T-style flow training

Leave `RUN_REPLAY_AND_FLOW=False` until body success/MPJPE is acceptable. Then change it to `True` in the configuration cell and run this cell. It collects causal token replay and trains the 1.76M-parameter text-conditioned flow model.

In [ ]:
REPLAY_DIR = DRIVE_WORK / 'replays' / selection_name
FLOW_DIR = DRIVE_WORK / 'runs' / selection_name / 'flow_text'
if RUN_REPLAY_AND_FLOW:
    subprocess.run([
        sys.executable, '-m', 'mini_groot_sonic.tools.collect_replay',
        '--motions', str(STAGE2_DATA), '--config', str(COLAB_CONFIG), '--mjcf', str(MJCF),
        '--checkpoint', str(BODY_CHECKPOINT), '--mode', 'policy',
        '--out', str(REPLAY_DIR), '--limit', str(REPLAY_EPISODES),
        '--seed', str(SEED), '--device', 'cuda:0',
    ], cwd=REPO_DIR, check=True)
    flow_cmd = [
        sys.executable, '-m', 'mini_groot_sonic.tools.train_flow',
        '--replays', str(REPLAY_DIR), '--config', str(COLAB_CONFIG), '--out', str(FLOW_DIR),
        '--device', 'cuda:0', '--epochs', str(FLOW_EPOCHS),
        '--batch-size', '64', '--workers', '2',
    ]
    if RESUME_EXISTING_RUNS:
        flow_checkpoints = sorted(FLOW_DIR.glob('flow_[0-9]*.pt'))
        if flow_checkpoints:
            flow_cmd += ['--resume', str(flow_checkpoints[-1])]
    subprocess.run(flow_cmd, cwd=REPO_DIR, check=True)
else:
    print('Skipped replay/flow. Enable RUN_REPLAY_AND_FLOW after body validation passes.')


## What to inspect before scaling

1. Stage-1 and stage-2 held-out success should trend upward.
2. Root error, MPJPE, undesired contacts, and action rate should trend downward.
3. FSQ occupancy should not collapse near zero; saturation should remain modest.
4. Only then increase to 128 environments, 128–256 records, or enable the upper flow model.
5. Do not add flips, jumps, stairs, or arbitrary target perturbations until basic tracking is robust.